# IQL Robustness Under Distribution Shift — Hopper
**CMPE 260 — Group 6 | Shloak Aggarwal (018189938)**

This notebook evaluates baseline IQL (DoubleCritic) on `hopper-medium-v2` under two types of controlled distribution shift:

1. **Gravity shift** — scales MuJoCo gravity at evaluation time: `[0.5×, 1.0×, 2.0×]`
2. **Observation noise** — adds Gaussian noise to observations: `σ ∈ {0.01, 0.1, 0.3}`

The policy is trained once on the unmodified dataset and then evaluated under each shift level without retraining.

**Outputs:**
- `results_shift_gravity_hopper.csv`
- `results_shift_noise_hopper.csv`

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q flax optax
!pip install -q tensorflow-probability[jax]
!pip install -q mujoco "gymnasium[mujoco]"
!pip install -q h5py tqdm matplotlib pandas
print('Done')

## Cell 2 — Verify JAX

In [ ]:
import jax
import jax.numpy as jnp
import flax
import optax
print('JAX:', jax.__version__)
print('Flax:', flax.__version__)
print('Devices:', jax.devices())

## Cell 3 — Clone Repo

In [ ]:
import os, sys
if not os.path.exists('/content/iql-robustness-analysis'):
    !git clone https://github.com/shloakk/iql-robustness-analysis /content/iql-robustness-analysis
%cd /content/iql-robustness-analysis
sys.path.insert(0, '/content/iql-robustness-analysis')
print('Repo ready')

## Cell 4 — Download and Load Dataset

In [ ]:
import h5py
import numpy as np

!wget -q -O hopper-medium-v2.hdf5 "https://rail.eecs.berkeley.edu/datasets/offline_rl/gym_mujoco_v2/hopper_medium-v2.hdf5"

with h5py.File('hopper-medium-v2.hdf5', 'r') as f:
    observations      = f['observations'][:].astype(np.float32)
    next_observations = f['next_observations'][:].astype(np.float32)
    actions           = f['actions'][:].astype(np.float32)
    rewards           = f['rewards'][:].astype(np.float32)
    terminals         = f['terminals'][:].astype(np.float32)

masks = 1.0 - terminals
eps   = 1e-5
actions = np.clip(actions, -(1-eps), (1-eps))

print(f'Transitions: {len(observations):,}')
print(f'obs shape:   {observations.shape}')
print(f'act shape:   {actions.shape}')

## Cell 5 — Normalize Rewards and Scoring Constants

In [ ]:
ep_returns, ep_ret = [], 0.0
for r, t in zip(rewards, terminals):
    ep_ret += r
    if t:
        ep_returns.append(ep_ret)
        ep_ret = 0.0

min_ret, max_ret = min(ep_returns), max(ep_returns)
rewards_norm = rewards / (max_ret - min_ret) * 1000.0

# D4RL normalization constants for hopper-medium-v2
RANDOM_SCORE = 20.272305
EXPERT_SCORE = 3234.3

def normalized_score(raw):
    return (raw - RANDOM_SCORE) / (EXPERT_SCORE - RANDOM_SCORE) * 100.0

print(f'Episode returns — min: {min_ret:.1f}, max: {max_ret:.1f}, count: {len(ep_returns)}')

## Cell 6 — Dataset Sampler

In [ ]:
import collections

Batch = collections.namedtuple(
    'Batch', ['observations', 'actions', 'rewards', 'masks', 'next_observations'])

class D4RLDataset:
    def __init__(self, obs, acts, rews, masks, next_obs):
        self.observations      = obs
        self.actions           = acts
        self.rewards           = rews
        self.masks             = masks
        self.next_observations = next_obs
        self.size = len(obs)

    def sample(self, batch_size):
        idx = np.random.randint(self.size, size=batch_size)
        return Batch(
            observations=self.observations[idx],
            actions=self.actions[idx],
            rewards=self.rewards[idx],
            masks=self.masks[idx],
            next_observations=self.next_observations[idx]
        )

dataset = D4RLDataset(observations, actions, rewards_norm, masks, next_observations)
print(f'Dataset ready — {dataset.size:,} transitions')

## Cell 7 — Network Definitions

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from typing import Sequence, Callable


class MLP(nn.Module):
    hidden_dims: Sequence[int]
    activations: Callable = nn.relu
    activate_final: bool = False

    @nn.compact
    def __call__(self, x):
        for i, size in enumerate(self.hidden_dims):
            x = nn.Dense(size)(x)
            if i + 1 < len(self.hidden_dims) or self.activate_final:
                x = self.activations(x)
        return x


class ValueNet(nn.Module):
    hidden_dims: Sequence[int]

    @nn.compact
    def __call__(self, obs):
        return jnp.squeeze(MLP((*self.hidden_dims, 1))(obs), -1)


class SingleCritic(nn.Module):
    hidden_dims: Sequence[int]

    @nn.compact
    def __call__(self, obs, act):
        x = jnp.concatenate([obs, act], -1)
        return jnp.squeeze(MLP((*self.hidden_dims, 1))(x), -1)


class DoubleCritic(nn.Module):
    hidden_dims: Sequence[int]

    @nn.compact
    def __call__(self, obs, act):
        q1 = SingleCritic(self.hidden_dims)(obs, act)
        q2 = SingleCritic(self.hidden_dims)(obs, act)
        return q1, q2


class Actor(nn.Module):
    hidden_dims: Sequence[int]
    action_dim: int
    log_std_min: float = -5.0
    log_std_max: float = 2.0

    @nn.compact
    def __call__(self, obs):
        x    = MLP(self.hidden_dims, activate_final=True)(obs)
        mean = nn.Dense(self.action_dim)(x)
        mean = nn.tanh(mean)
        log_std = self.param('log_std', nn.initializers.zeros, (self.action_dim,))
        log_std = jnp.clip(log_std, self.log_std_min, self.log_std_max)
        return mean, jnp.exp(log_std)


print('Networks defined')

## Cell 8 — Training State and Update Functions

In [ ]:
from flax.training import train_state
import functools


class TrainState(train_state.TrainState):
    pass


def create_train_state(model, dummy_inputs, lr):
    params = model.init(jax.random.PRNGKey(0), *dummy_inputs)['params']
    tx = optax.adam(lr)
    return TrainState.create(apply_fn=model.apply, params=params, tx=tx)


def expectile_loss(diff, expectile=0.7):
    weight = jnp.where(diff > 0, expectile, 1 - expectile)
    return (weight * diff ** 2).mean()


@jax.jit
def update_value(value_state, critic_state, batch, expectile=0.7):
    q1, q2 = critic_state.apply_fn({'params': critic_state.params},
                                    batch.observations, batch.actions)
    q = jnp.minimum(q1, q2)

    def loss_fn(params):
        v = value_state.apply_fn({'params': params}, batch.observations)
        return expectile_loss(q - v, expectile), v.mean()

    (loss, v_mean), grads = jax.value_and_grad(loss_fn, has_aux=True)(value_state.params)
    return value_state.apply_gradients(grads=grads), loss, v_mean


@jax.jit
def update_critic(critic_state, target_value_state, batch, discount=0.99):
    next_v = target_value_state.apply_fn({'params': target_value_state.params},
                                          batch.next_observations)
    target_q = batch.rewards + discount * batch.masks * next_v

    def loss_fn(params):
        q1, q2 = critic_state.apply_fn({'params': params},
                                        batch.observations, batch.actions)
        loss = ((q1 - target_q) ** 2 + (q2 - target_q) ** 2).mean()
        return loss, (q1.mean(), q2.mean())

    (loss, _), grads = jax.value_and_grad(loss_fn, has_aux=True)(critic_state.params)
    return critic_state.apply_gradients(grads=grads), loss


@jax.jit
def update_actor(actor_state, critic_state, value_state, batch, temperature=3.0):
    v = value_state.apply_fn({'params': value_state.params}, batch.observations)
    q1, q2 = critic_state.apply_fn({'params': critic_state.params},
                                    batch.observations, batch.actions)
    q = jnp.minimum(q1, q2)
    adv     = q - v
    weights = jnp.minimum(jnp.exp(adv * temperature), 100.0)

    def loss_fn(params):
        mean, std = actor_state.apply_fn({'params': params}, batch.observations)
        log_prob = -0.5 * (((batch.actions - mean) / std) ** 2
                           + 2 * jnp.log(std)
                           + jnp.log(2 * jnp.pi))
        log_prob = log_prob.sum(-1)
        return -(weights * log_prob).mean()

    loss, grads = jax.value_and_grad(loss_fn)(actor_state.params)
    return actor_state.apply_gradients(grads=grads), loss


@jax.jit
def soft_update(source_params, target_params, tau=0.005):
    return jax.tree_util.tree_map(
        lambda s, t: tau * s + (1 - tau) * t,
        source_params, target_params)


print('Update functions defined')

## Cell 9 — Shift Wrappers

In [ ]:
import gymnasium as gym
import numpy as np


class GravityWrapper(gym.Wrapper):
    """
    Wraps a MuJoCo environment and scales gravity by a given factor.
    Default MuJoCo gravity is -9.81 m/s^2 (along the z-axis).

    Args:
        env: The base Gymnasium/MuJoCo environment.
        gravity_scale: Multiplier applied to default gravity.
                       0.5x = low gravity, 1.0x = normal, 2.0x = high gravity.
    """
    def __init__(self, env, gravity_scale: float = 1.0):
        super().__init__(env)
        self.gravity_scale = gravity_scale
        self._default_gravity = env.unwrapped.model.opt.gravity.copy()

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.env.unwrapped.model.opt.gravity[:] = (
            self._default_gravity * self.gravity_scale
        )
        return obs, info

    def step(self, action):
        return self.env.step(action)


class ObservationNoise(gym.ObservationWrapper):
    """
    Adds Gaussian noise to observations to simulate sensor noise.

    Args:
        env: A gymnasium environment.
        noise_std: Standard deviation of the Gaussian noise.
                   0.0 = no noise (baseline behavior).
    """
    def __init__(self, env: gym.Env, noise_std: float = 0.0):
        super().__init__(env)
        self.noise_std = noise_std

    def observation(self, observation: np.ndarray) -> np.ndarray:
        if self.noise_std > 0:
            noise = np.random.normal(
                0, self.noise_std, size=observation.shape
            ).astype(observation.dtype)
            return observation + noise
        return observation


print('Shift wrappers ready')

## Cell 10 — Initialize Baseline IQL Agent

In [ ]:
import gymnasium as gym

env     = gym.make('Hopper-v4')
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
env.close()

HIDDEN    = (256, 256)
LR        = 3e-4
dummy_obs = jnp.zeros((1, obs_dim))
dummy_act = jnp.zeros((1, act_dim))

actor_state  = create_train_state(Actor(HIDDEN, act_dim), [dummy_obs], LR)
critic_state = create_train_state(DoubleCritic(HIDDEN), [dummy_obs, dummy_act], LR)
value_state  = create_train_state(ValueNet(HIDDEN), [dummy_obs], LR)
target_value_params = value_state.params

print(f'obs_dim={obs_dim}, act_dim={act_dim}')
print('Baseline IQL agent initialized')

## Cell 11 — Evaluation Functions

In [ ]:
import numpy as np


def evaluate(actor_state, env_name='Hopper-v4', num_episodes=10,
             gravity_scale=None, noise_std=None):
    """Evaluate policy under optional distribution shift.

    Args:
        actor_state: Trained Flax actor train state.
        env_name: Gymnasium environment name.
        num_episodes: Number of evaluation episodes.
        gravity_scale: If set, wrap env with GravityWrapper at this scale.
        noise_std: If set, wrap env with ObservationNoise at this std.

    Returns:
        Mean raw return over episodes.
    """
    env = gym.make(env_name)
    if gravity_scale is not None:
        env = GravityWrapper(env, gravity_scale=gravity_scale)
    if noise_std is not None:
        env = ObservationNoise(env, noise_std=noise_std)

    returns = []
    for _ in range(num_episodes):
        obs, _ = env.reset()
        done, ep_return = False, 0.0
        while not done:
            obs_j = jnp.array(obs[np.newaxis], dtype=jnp.float32)
            mean, _ = actor_state.apply_fn({'params': actor_state.params}, obs_j)
            action = np.clip(np.array(mean[0]), -1, 1)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward
        returns.append(ep_return)
    env.close()
    return float(np.mean(returns))


print('Evaluation function ready')

## Cell 12 — Train Baseline IQL (300k steps)

In [ ]:
import tqdm

MAX_STEPS     = 300_000
EVAL_INTERVAL = 50_000
BATCH_SIZE    = 256
EXPECTILE     = 0.7
TEMPERATURE   = 3.0
DISCOUNT      = 0.99
TAU           = 0.005

train_log = []

print('Training baseline IQL on hopper-medium-v2 ...')
for i in tqdm.tqdm(range(1, MAX_STEPS + 1), smoothing=0.1):
    batch = dataset.sample(BATCH_SIZE)
    jbatch = Batch(
        observations=jnp.array(batch.observations),
        actions=jnp.array(batch.actions),
        rewards=jnp.array(batch.rewards),
        masks=jnp.array(batch.masks),
        next_observations=jnp.array(batch.next_observations)
    )

    target_value_state = value_state.replace(params=target_value_params)

    value_state, v_loss, _  = update_value(value_state, critic_state, jbatch, EXPECTILE)
    critic_state, c_loss    = update_critic(critic_state, target_value_state, jbatch, DISCOUNT)
    actor_state, a_loss     = update_actor(actor_state, critic_state, value_state, jbatch, TEMPERATURE)
    target_value_params     = soft_update(value_state.params, target_value_params, TAU)

    if i % EVAL_INTERVAL == 0:
        raw    = evaluate(actor_state, num_episodes=10)
        norm   = normalized_score(raw)
        train_log.append({'step': i, 'raw_return': raw, 'normalized_score': norm})
        print(f'Step {i:>7,} | Raw: {raw:>7.1f} | Norm Score: {norm:.2f}')

print('\nTraining complete.')

## Cell 13 — Evaluate Under Gravity Shift

In [ ]:
GRAVITY_LEVELS = [0.5, 1.0, 2.0]
NUM_EVAL_EPS   = 20  # More episodes for stable shift estimates

gravity_results = []

print('Evaluating under gravity shift ...')
print(f'{"Scale":>8}  {"Raw Return":>12}  {"Norm Score":>12}  {"Δ vs Baseline":>14}')
print('-' * 55)

baseline_norm = normalized_score(evaluate(actor_state, num_episodes=NUM_EVAL_EPS))

for scale in GRAVITY_LEVELS:
    raw  = evaluate(actor_state, num_episodes=NUM_EVAL_EPS, gravity_scale=scale)
    norm = normalized_score(raw)
    delta = norm - baseline_norm
    gravity_results.append({
        'shift_type': 'gravity',
        'shift_level': scale,
        'raw_return': raw,
        'normalized_score': norm,
        'delta_vs_baseline': delta
    })
    print(f'{scale:>8.1f}x  {raw:>12.2f}  {norm:>12.2f}  {delta:>+14.2f}')

## Cell 14 — Evaluate Under Observation Noise

In [ ]:
NOISE_LEVELS = [0.01, 0.1, 0.3]

noise_results = []

print('Evaluating under observation noise ...')
print(f'{"Noise σ":>8}  {"Raw Return":>12}  {"Norm Score":>12}  {"Δ vs Baseline":>14}')
print('-' * 55)

for std in NOISE_LEVELS:
    raw  = evaluate(actor_state, num_episodes=NUM_EVAL_EPS, noise_std=std)
    norm = normalized_score(raw)
    delta = norm - baseline_norm
    noise_results.append({
        'shift_type': 'obs_noise',
        'shift_level': std,
        'raw_return': raw,
        'normalized_score': norm,
        'delta_vs_baseline': delta
    })
    print(f'{std:>8.2f}   {raw:>12.2f}  {norm:>12.2f}  {delta:>+14.2f}')

## Cell 15 — Combined Results Table

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_gravity = pd.DataFrame(gravity_results)
df_noise   = pd.DataFrame(noise_results)

print('=== Gravity Shift Results (hopper-medium-v2) ===')
print(df_gravity[['shift_level','raw_return','normalized_score','delta_vs_baseline']].to_string(index=False))
print()
print('=== Observation Noise Results (hopper-medium-v2) ===')
print(df_noise[['shift_level','raw_return','normalized_score','delta_vs_baseline']].to_string(index=False))
print()
print(f'Baseline (no shift): {baseline_norm:.2f}')

# --- plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.axhline(baseline_norm, color='gray', linestyle='--', label='Baseline (no shift)')
ax1.plot(df_gravity['shift_level'], df_gravity['normalized_score'],
         marker='o', linewidth=2, color='steelblue', label='Gravity shift')
ax1.set_xlabel('Gravity Scale')
ax1.set_ylabel('Normalized Score')
ax1.set_title('IQL Robustness — Gravity Shift\nhopper-medium-v2')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.axhline(baseline_norm, color='gray', linestyle='--', label='Baseline (no shift)')
ax2.plot(df_noise['shift_level'], df_noise['normalized_score'],
         marker='s', linewidth=2, color='darkorange', label='Obs noise')
ax2.set_xlabel('Noise σ')
ax2.set_ylabel('Normalized Score')
ax2.set_title('IQL Robustness — Observation Noise\nhopper-medium-v2')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results_shift_hopper.png', dpi=150)
plt.show()
print('Plot saved as results_shift_hopper.png')

## Cell 16 — Save Results to CSV

In [ ]:
df_gravity.to_csv('results_shift_gravity_hopper.csv', index=False)
df_noise.to_csv('results_shift_noise_hopper.csv', index=False)
print('Saved: results_shift_gravity_hopper.csv')
print('Saved: results_shift_noise_hopper.csv')

## Summary — Report Section

### Shift Design

We evaluate robustness by applying environment modifications **only at test time** — the IQL policy is trained once on the unperturbed `hopper-medium-v2` dataset and then evaluated under each shift level without any fine-tuning.

**Gravity shift (`GravityWrapper`)** scales MuJoCo's gravity vector `[0, 0, −9.81]` by a constant factor at every `reset()`. This tests whether the policy generalizes to altered dynamics (e.g., a heavier or lighter robot).

| Scale | Interpretation |
|-------|----------------|
| 0.5×  | Half gravity — easier to stay upright, but gait is very different |
| 1.0×  | No shift — matches training conditions |
| 2.0×  | Double gravity — harder to balance, higher joint torques needed |

**Observation noise (`ObservationNoise`)** adds i.i.d. Gaussian noise $\mathcal{N}(0, \sigma^2)$ to each observation dimension before passing it to the policy. This models sensor noise or partial observability.

| σ    | Interpretation |
|------|----------------|
| 0.01 | Mild noise — nearly imperceptible |
| 0.1  | Moderate noise — comparable to observation scale |
| 0.3  | Heavy noise — significant corruption |

### Robustness Metric

We report **normalized score** (D4RL convention) and the **robustness drop** Δ = score(shifted) − score(baseline). A larger negative Δ indicates a more brittle policy.

### Initial Results

*(See CSVs and plot above — populated after running the notebook on Colab GPU.)*